# Mitra Regressor — DIMER end-to-end regression tutorial

**Profile:** `E2E`  
**Notebook specification:** `1.0`  
**Capability:** tabular regression with the pinned `autogluon/mitra-regressor` release.

Mitra is a tabular foundation model. In the default workflow, AutoGluon gives Mitra labelled support rows as in-context information and calls `fit()`, but **no gradient update occurs** because `fine_tune=False`. Optional GPU fine-tuning is a separate adaptation mode and does update model weights.

The upstream AutoGluon/Mitra release supplies the pretrained model and Mitra integration. This repository adds the DIMER-facing regression contract, user-facing validation rules, immutable model provenance, split-integrity checks, artifact packaging, and a public notebook API in `mitra_pipeline`.

**By the end of this notebook you will be able to:**
- acquire and byte-verify the immutable Mitra Regressor checkpoint;
- validate the bundled sample or your own regression table through the repository API;
- preserve leakage-aware partitions or use a seeded random holdout only when IID rows are a defensible assumption;
- evaluate Mitra using MAE, RMSE, R², and executable constant/classical baselines;
- optionally fine-tune Mitra without using the independent test set for selection;
- score genuinely new rows;
- export the reusable AutoGluon predictor with provenance and a file-digest manifest; and
- reload the serialized artifact from a fresh directory and verify prediction equivalence.

**Capability boundaries.** This pipeline predicts one continuous numeric value per row. It is not a general time-series forecaster, classifier, causal model, or uncertainty-estimation system. Point predictions do **not** include calibrated per-prediction intervals.

**Evidence boundary.** Successful execution proves the demonstrated code path worked for the exact model, data, and runtime recorded by this run. Tutorial/sample metrics are not benchmark evidence and do not establish production fitness.

No DIMER Workbench access is required. Data stays in the notebook runtime; it is not uploaded to DIMER. Do not place confidential, restricted, or sensitive data in Colab unless that environment is authorized for it.

**References:** [Repository README](https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/README.md) · [Model card](https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/MODEL_CARD.md) · [Upstream model](https://huggingface.co/autogluon/mitra-regressor) · [Paper](https://arxiv.org/abs/2510.21204)


## 1. Install and identify the runtime

The notebook clones this repository and imports its public `mitra_pipeline` API so core validation, fit, inference, and artifact checks are repository code rather than notebook reimplementations. Dependencies come from the repository's pinned tutorial requirements.

The default pretrained/in-context path does not require a GPU. GPU is required only when `RUN_FINE_TUNING=True`. AutoGluon may replace Colab's preinstalled PyTorch; if PyTorch was already imported and the installed version changes, restart the runtime and run from the top.

**Remaining variability after seeding:** Mitra/AutoGluon, accelerator kernels, library implementations, and time-limited fine-tuning can introduce run-to-run variation. The seed controls the notebook's explicit splits/sampling and the seed passed into Mitra; it is not a claim of bitwise determinism across hardware.

**What to look for:** Python, AutoGluon, PyTorch, CUDA availability, and the exact repository commit used by this run.


In [ ]:
import importlib.metadata as importlib_metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/mitra-regressor-pipeline.git"
REPO_REF = os.environ.get("DIMER_REPO_REF", "main").strip() or "main"
REPO_DIR = Path("/content/mitra-regressor-pipeline")

try:
    PREINSTALL_TORCH_VERSION = importlib_metadata.version("torch")
except importlib_metadata.PackageNotFoundError:
    PREINSTALL_TORCH_VERSION = None
TORCH_WAS_IMPORTED = "torch" in sys.modules

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

%pip install -q -r /content/mitra-regressor-pipeline/tutorials/requirements-colab.txt

INSTALLED_TORCH_VERSION = importlib_metadata.version("torch")
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError(
        "pip changed PyTorch after it had already been imported. "
        "Use Runtime → Restart session, then run the notebook top-to-bottom."
    )

sys.path.insert(0, str(REPO_DIR))
import lightgbm
import torch
import mitra_pipeline as mp

AUTOGLUON_VERSION = importlib_metadata.version("autogluon.tabular")
print("Profile: E2E")
print("Notebook spec: 1.0")
print("Python:", sys.version.split()[0])
print("AutoGluon:", AUTOGLUON_VERSION)
print("LightGBM:", lightgbm.__version__)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Repository commit:", REPO_COMMIT)
print("Model:", mp.MODEL_ID)
print("Pinned model revision:", mp.PINNED_REVISION)


## 2. Acquire, verify, and lock the model

The default path downloads `model.safetensors` and `config.json` from the exact immutable Hugging Face commit recorded by this repository. Both files are SHA-256 verified and staged into an offline Hugging Face snapshot before AutoGluon sees them.

`DIMER ZIP` accepts only the Notebook-Spec-v1 package format: the archive must contain root-level `dimer-model-manifest.json`, `model.safetensors`, and `config.json`. The manifest must identify this model and revision and list every package file with its exact size and SHA-256. Legacy weight-only ZIPs are rejected rather than silently treated as verified packages.

The Mitra implementation used here comes from the pinned installed AutoGluon package. The notebook does not execute Python code from the model repository.

**What to look for:** the exact model id/revision, both expected digests, and the path of the verified offline snapshot.


In [ ]:
import urllib.request

WORKDIR = Path("/content/mitra-tutorial")
MODEL_DIR = WORKDIR / "model"
HF_HOME = WORKDIR / "hf"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)

MODEL_SOURCE = "Pinned upstream"  # @param ["Pinned upstream", "DIMER ZIP"]
EXPECTED_DIMER_ZIP_SHA256 = ""   # @param {type:"string"}
NETWORK_TIMEOUT_SECONDS = 30

weights_path = MODEL_DIR / "model.safetensors"
config_path = MODEL_DIR / "config.json"

def fetch_pinned(filename, destination):
    url = (
        f"https://huggingface.co/{mp.MODEL_ID}/resolve/"
        f"{mp.PINNED_REVISION}/{filename}?download=true"
    )
    with urllib.request.urlopen(url, timeout=NETWORK_TIMEOUT_SECONDS) as response:
        destination.write_bytes(response.read())

if MODEL_SOURCE == "Pinned upstream":
    fetch_pinned("model.safetensors", weights_path)
    fetch_pinned("config.json", config_path)
else:
    supplied = os.environ.get("DIMER_MODEL_ZIP_PATH", "").strip()
    if supplied:
        dimer_zip = Path(supplied)
    else:
        from google.colab import files
        uploaded = files.upload()
        candidates = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith(".zip")]
        if len(candidates) != 1:
            raise RuntimeError("Upload exactly one DIMER model ZIP.")
        name, payload = candidates[0]
        dimer_zip = WORKDIR / Path(name).name
        dimer_zip.write_bytes(payload)
    weights_path, config_path, package_manifest = mp.validate_dimer_model_package(
        dimer_zip,
        WORKDIR / "dimer-model-package",
        expected_archive_sha256=EXPECTED_DIMER_ZIP_SHA256 or None,
    )
    print("DIMER package schema:", package_manifest["schema_version"])

SNAPSHOT_PATH = mp.stage_verified_hf_snapshot(
    weights_path,
    config_path,
    hf_home=HF_HOME,
)

print("Model:", mp.MODEL_ID)
print("Revision:", mp.PINNED_REVISION)
print("Weights SHA-256:", mp.WEIGHTS_SHA256)
print("Config SHA-256:", mp.CONFIG_SHA256)
print("Verified offline snapshot:", SNAPSHOT_PATH)


## 3. Load and validate sample or BYOD data

The default sample is the pinned FreshRetailNet-derived regression package. Its provided `train.csv`, `val.csv`, and `test.csv` partition membership is preserved. They are a purged chronological split with an embargo and are tutorial/sanity data, **not benchmark evidence**. To keep the literal CPU-default path practical for an interactive tutorial and for clean release execution, the notebook deterministically draws a bounded smoke subset from within those already-separated partitions: 512 training rows and 256 rows from each evaluation split. Increase the two sample-row form values to use more or all rows; BYOD paths are unaffected by this smoke-only cap.

For BYOD, the expected schema is a CSV with one finite numeric target column plus 1–500 feature columns. Training requires at least 50 usable labelled rows. The notebook rejects duplicate raw headers, missing targets, non-numeric/infinite targets, missing features, and unsupported feature counts. Missing-target row drops and the 10,000-row Mitra training cap are reported.

- **Upload CSV** uses a seeded random holdout and assumes rows are sufficiently IID for that split to be meaningful.
- **Upload pre-split train/val/test** preserves your partitions and is the correct route for temporal, grouped, panel, embargoed, patient/device-level, or otherwise leakage-sensitive data.
- Optional upload dialogs are disabled on the default sample path.

Exact record overlap across provided partitions is detected and surfaced. A nonzero overlap is a leakage warning requiring investigation.


In [ ]:
import hashlib
import io
import json
import zipfile

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_SOURCE = "Sample dataset (FreshRetailNet)"  # @param ["Sample dataset (FreshRetailNet)", "Upload CSV", "Upload pre-split train/val/test"]
TARGET_COLUMN = "target"                         # @param {type:"string"}
DROP_COLUMNS = ""                                # @param {type:"string"}
VALIDATION_SPLIT = 0.20                          # @param {type:"number"}
SEED = 42                                        # @param {type:"integer"}
SAMPLE_TRAIN_ROWS = 512                         # @param {type:"integer"}
SAMPLE_EVAL_ROWS = 256                          # @param {type:"integer"}

SAMPLE_REVISION = "5625a9eeca94b8c72b9ad1ec78d07ecbaa720903"
SAMPLE_URL = (
    "https://raw.githubusercontent.com/kurtvalcorza/mitra-regressor-pipeline/"
    f"{SAMPLE_REVISION}/examples/sample-data/freshretailnet-h7.zip"
)
SAMPLE_CARD_URL = (
    "https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/"
    f"{SAMPLE_REVISION}/examples/sample-data/DATASET_CARD.md"
)

drop_columns = [c.strip() for c in DROP_COLUMNS.split(",") if c.strip() and c.strip() != TARGET_COLUMN]
using_sample = DATA_SOURCE == "Sample dataset (FreshRetailNet)"
using_presplit = DATA_SOURCE in {
    "Sample dataset (FreshRetailNet)",
    "Upload pre-split train/val/test",
}
test_data = None
DATA_DIGEST = None

def report_validation(label, report):
    if report["dropped_missing_target_rows"]:
        print(f"⚠ {label}: dropped {report['dropped_missing_target_rows']:,} row(s) with missing target.")
    if report["exact_duplicate_rows"]:
        print(f"⚠ {label}: {report['exact_duplicate_rows']:,} exact duplicate row(s) detected.")

def validate_split(frame, label, require_variation=False, min_rows=2):
    clean, features, report = mp.validate_labeled_frame(
        frame,
        TARGET_COLUMN,
        name=label,
        drop_columns=drop_columns,
        min_rows=min_rows,
        require_variation=require_variation,
    )
    report_validation(label, report)
    return clean, features

if using_sample:
    with urllib.request.urlopen(SAMPLE_URL, timeout=NETWORK_TIMEOUT_SECONDS) as response:
        sample_payload = response.read()
    DATA_DIGEST = hashlib.sha256(sample_payload).hexdigest()
    with zipfile.ZipFile(io.BytesIO(sample_payload)) as zf:
        names = {Path(name).name: name for name in zf.namelist() if not name.endswith("/")}
        required = {"train.csv", "val.csv", "test.csv"}
        missing = sorted(required - set(names))
        if missing:
            raise RuntimeError(f"Sample ZIP missing: {missing}")
        train_data = mp.read_csv_bytes(zf.read(names["train.csv"]), "train.csv")
        holdout_data = mp.read_csv_bytes(zf.read(names["val.csv"]), "val.csv")
        test_data = mp.read_csv_bytes(zf.read(names["test.csv"]), "test.csv")

    def deterministic_smoke_subset(frame, rows, seed):
        if rows <= 0:
            raise ValueError("Sample row controls must be positive integers.")
        if len(frame) <= rows:
            return frame.reset_index(drop=True)
        return frame.sample(n=rows, random_state=seed).sort_index().reset_index(drop=True)

    train_data = deterministic_smoke_subset(train_data, SAMPLE_TRAIN_ROWS, SEED)
    holdout_data = deterministic_smoke_subset(holdout_data, SAMPLE_EVAL_ROWS, SEED + 1)
    test_data = deterministic_smoke_subset(test_data, SAMPLE_EVAL_ROWS, SEED + 2)
    print("Default smoke subset:", {"train": len(train_data), "holdout": len(holdout_data), "test": len(test_data)})
    TARGET_COLUMN = "target"
    print("✓ Loaded pinned FreshRetailNet tutorial sample.")
    print("Sample revision:", SAMPLE_REVISION)
    print("Sample SHA-256:", DATA_DIGEST)
    print("Dataset card:", SAMPLE_CARD_URL)
elif DATA_SOURCE == "Upload pre-split train/val/test":
    paths = {
        "train.csv": os.environ.get("DIMER_TRAIN_CSV", "").strip(),
        "val.csv": os.environ.get("DIMER_VAL_CSV", "").strip(),
        "test.csv": os.environ.get("DIMER_TEST_CSV", "").strip(),
    }
    if all(paths.values()):
        payloads = {name: Path(path).read_bytes() for name, path in paths.items()}
    else:
        from google.colab import files
        uploaded = files.upload()
        by_base = {Path(name).name.lower(): payload for name, payload in uploaded.items()}
        required = {"train.csv", "val.csv", "test.csv"}
        missing = sorted(required - set(by_base))
        if missing:
            raise RuntimeError(f"Upload train.csv, val.csv, and test.csv together. Missing: {missing}")
        payloads = {name: by_base[name] for name in required}
    train_data = mp.read_csv_bytes(payloads["train.csv"], "train.csv")
    holdout_data = mp.read_csv_bytes(payloads["val.csv"], "val.csv")
    test_data = mp.read_csv_bytes(payloads["test.csv"], "test.csv")
    digest_record = {name: hashlib.sha256(payload).hexdigest() for name, payload in sorted(payloads.items())}
    DATA_DIGEST = hashlib.sha256(json.dumps(digest_record, sort_keys=True).encode()).hexdigest()
    print("✓ Loaded user-provided pre-split partitions without re-splitting.")
else:
    supplied = os.environ.get("DIMER_TRAIN_CSV", "").strip()
    if supplied:
        payload = Path(supplied).read_bytes()
    else:
        from google.colab import files
        uploaded = files.upload()
        csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith(".csv")]
        if len(csvs) != 1:
            raise RuntimeError("Upload exactly one labelled CSV.")
        payload = csvs[0][1]
    DATA_DIGEST = hashlib.sha256(payload).hexdigest()
    data = mp.read_csv_bytes(payload, "uploaded CSV")
    print("⚠ Upload CSV uses a seeded random holdout and assumes approximately IID rows.")

if using_presplit:
    train_data, features = validate_split(train_data, "train.csv", require_variation=True, min_rows=mp.MIN_TRAIN_ROWS)
    holdout_data, val_features = validate_split(holdout_data, "val.csv")
    test_data, test_features = validate_split(test_data, "test.csv")
    if set(val_features) != set(features) or set(test_features) != set(features):
        raise ValueError("train/val/test feature column names do not match.")
    ordered = features + [TARGET_COLUMN]
    holdout_data = holdout_data.reindex(columns=ordered)
    test_data = test_data.reindex(columns=ordered)
    overlaps = mp.split_overlap_report(
        {"train": train_data[ordered], "holdout": holdout_data[ordered], "test": test_data[ordered]}
    )
    print("Exact cross-split overlap counts:", overlaps)
    if any(overlaps.values()):
        print("⚠ Exact cross-split overlap detected. Investigate leakage before interpreting metrics.")
else:
    clean, features = validate_split(data, "uploaded CSV", require_variation=True, min_rows=mp.MIN_TRAIN_ROWS)
    if not 0.05 <= VALIDATION_SPLIT <= 0.40:
        raise ValueError("VALIDATION_SPLIT must be between 0.05 and 0.40.")
    train_data, holdout_data = train_test_split(
        clean,
        test_size=VALIDATION_SPLIT,
        random_state=SEED,
        shuffle=True,
    )
    if train_data[TARGET_COLUMN].nunique(dropna=True) < 2:
        raise ValueError("Training split has no target variation.")
    overlaps = mp.split_overlap_report({"train": train_data, "holdout": holdout_data})
    print("Exact cross-split overlap counts:", overlaps)

train_data, cap_report = mp.cap_training_rows(
    train_data,
    TARGET_COLUMN,
    seed=SEED,
)
if cap_report["applied"]:
    print(f"⚠ Training rows capped from {cap_report['before']:,} to {cap_report['after']:,} using seed={SEED}.")

FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
if len(FEATURE_COLUMNS) > 100:
    print("⚠ Above Mitra's particularly strong reported ≤100-feature regime.")
if len(train_data) > 5_000:
    print("⚠ Above Mitra's particularly strong reported ≤5,000-sample regime.")

summary_frames = [("train", train_data), ("holdout", holdout_data)]
if test_data is not None:
    summary_frames.append(("test", test_data))
display(pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "target_mean": frame[TARGET_COLUMN].mean(),
        "target_std": frame[TARGET_COLUMN].std(),
        "target_min": frame[TARGET_COLUMN].min(),
        "target_max": frame[TARGET_COLUMN].max(),
    }
    for name, frame in summary_frames
]))
print("Features:", len(FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Data identity SHA-256:", DATA_DIGEST)


## 4. Evaluate pretrained Mitra and executable baselines

This stage uses the repository's public `fit_mitra_predictor()` and prediction path.

With `RUN_FINE_TUNING=False`, `fit()` registers the labelled support context and model configuration; the verified checkpoint weights are unchanged. With fine-tuning enabled on a GPU, weights are adapted for the requested number of steps, subject to the time limit.

For pre-split data, the validation partition is the **selection holdout** and `test.csv` is an **independent test**. The independent test is never used to choose between pretrained and fine-tuned variants.

Regression metrics:
- **MAE**: typical absolute error in the target's units;
- **RMSE**: error in the target's units with greater penalty on large misses;
- **R²**: fraction of variance explained relative to a constant mean predictor; interpret with MAE/RMSE rather than alone.

The notebook also computes literal mean and median `DummyRegressor` baselines, plus LightGBM and Random Forest on the same training/evaluation partitions. These are current-run tutorial metrics, not quoted upstream benchmarks.


In [ ]:
import gc
import time

from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

EVAL_METRIC = "mean_absolute_error"  # @param ["mean_absolute_error", "root_mean_squared_error"]
BASELINE_TIME_LIMIT = 300            # @param {type:"integer"}
RUN_FINE_TUNING = False              # @param {type:"boolean"}
FINE_TUNE_STEPS = 50                 # @param {type:"integer"}
FINE_TUNE_TIME_LIMIT = 600           # @param {type:"integer"}
MAX_MEMORY_USAGE_RATIO = 1.10        # @param {type:"number"}
MIN_SELECTION_HOLDOUT_ROWS = 50

BASELINE_PATH = WORKDIR / "mitra-pretrained"
FINETUNED_PATH = WORKDIR / "mitra-finetuned"
for path in (BASELINE_PATH, FINETUNED_PATH):
    shutil.rmtree(path, ignore_errors=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def evaluate_predictor(predictor, frame):
    raw = predictor.evaluate(frame, auxiliary_metrics=True, silent=True)
    return mp.normalize_autogluon_regression_metrics(raw)

baseline_predictor = mp.fit_mitra_predictor(
    train_data,
    target_column=TARGET_COLUMN,
    eval_metric=EVAL_METRIC,
    path=BASELINE_PATH,
    fine_tune=False,
    time_limit=BASELINE_TIME_LIMIT,
    seed=SEED,
    max_memory_usage_ratio=MAX_MEMORY_USAGE_RATIO,
)
baseline_holdout = evaluate_predictor(baseline_predictor, holdout_data)
baseline_test = evaluate_predictor(baseline_predictor, test_data) if test_data is not None else None

finetuned_predictor = None
finetuned_holdout = None
finetuned_test = None
if RUN_FINE_TUNING:
    if not torch.cuda.is_available():
        raise RuntimeError("Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU.")
    finetuned_predictor = mp.fit_mitra_predictor(
        train_data,
        target_column=TARGET_COLUMN,
        eval_metric=EVAL_METRIC,
        path=FINETUNED_PATH,
        fine_tune=True,
        fine_tune_steps=FINE_TUNE_STEPS,
        time_limit=FINE_TUNE_TIME_LIMIT,
        seed=SEED,
        max_memory_usage_ratio=MAX_MEMORY_USAGE_RATIO,
    )
    finetuned_holdout = evaluate_predictor(finetuned_predictor, holdout_data)
    finetuned_test = evaluate_predictor(finetuned_predictor, test_data) if test_data is not None else None

from mitra_pipeline.tutorial_api import REGRESSION_LOWER_IS_BETTER

def better(candidate, reference, metric):
    if metric in REGRESSION_LOWER_IS_BETTER:
        return candidate[metric] < reference[metric]
    return candidate[metric] > reference[metric]

active_predictor = baseline_predictor
active_mode = "pretrained"
selection_basis = "default:pretrained"
if finetuned_predictor is not None:
    if len(holdout_data) < MIN_SELECTION_HOLDOUT_ROWS:
        selection_basis = f"default:pretrained; holdout-too-small:{len(holdout_data)}<{MIN_SELECTION_HOLDOUT_ROWS}"
        print("⚠ Holdout too small for automatic variant selection; keeping pretrained.")
    else:
        selection_basis = f"holdout:{EVAL_METRIC}"
        if better(finetuned_holdout, baseline_holdout, EVAL_METRIC):
            active_predictor = finetuned_predictor
            active_mode = "fine-tuned"
        print("Recommended predictor:", active_mode, "| selection basis:", selection_basis)

if finetuned_test is not None and baseline_test is not None:
    degraded = []
    for metric, before in baseline_test.items():
        after = finetuned_test.get(metric)
        if after is None:
            continue
        if metric in REGRESSION_LOWER_IS_BETTER and after > before:
            degraded.append(metric)
        elif metric not in REGRESSION_LOWER_IS_BETTER and after < before:
            degraded.append(metric)
    if degraded:
        print("⚠ Independent-test metrics worsened after fine-tuning:", degraded)
        print("Selection remains holdout-only; the independent test does not retroactively change the rule.")

# Executable constant baselines.
y_train = train_data[TARGET_COLUMN].to_numpy(dtype=float)
y_holdout = holdout_data[TARGET_COLUMN].to_numpy(dtype=float)
y_test = test_data[TARGET_COLUMN].to_numpy(dtype=float) if test_data is not None else None

baseline_rows = []
for strategy in ("mean", "median"):
    dummy = DummyRegressor(strategy=strategy)
    dummy.fit(np.zeros((len(y_train), 1)), y_train)
    pred_h = dummy.predict(np.zeros((len(y_holdout), 1)))
    row = {"model": f"Dummy-{strategy}", "split": "holdout", **mp.regression_metrics(y_holdout, pred_h)}
    baseline_rows.append(row)
    if y_test is not None:
        pred_t = dummy.predict(np.zeros((len(y_test), 1)))
        baseline_rows.append({"model": f"Dummy-{strategy}", "split": "test", **mp.regression_metrics(y_test, pred_t)})

# Train-fitted preprocessing for classical baselines; inference reuses this state and never refits on holdout/test.
cat_cols = [c for c in FEATURE_COLUMNS if not pd.api.types.is_numeric_dtype(train_data[c])]
num_cols = [c for c in FEATURE_COLUMNS if pd.api.types.is_numeric_dtype(train_data[c])]
num_imputer = SimpleImputer(strategy="median", keep_empty_features=True) if num_cols else None
cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1) if cat_cols else None

def fit_transform_tree(frame):
    parts = []
    if num_cols:
        parts.append(pd.DataFrame(
            num_imputer.fit_transform(frame[num_cols]),
            columns=num_cols,
            index=frame.index,
        ))
    if cat_cols:
        parts.append(pd.DataFrame(
            cat_encoder.fit_transform(frame[cat_cols].astype(str)),
            columns=cat_cols,
            index=frame.index,
        ))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]

def transform_tree(frame):
    parts = []
    if num_cols:
        parts.append(pd.DataFrame(
            num_imputer.transform(frame[num_cols]),
            columns=num_cols,
            index=frame.index,
        ))
    if cat_cols:
        parts.append(pd.DataFrame(
            cat_encoder.transform(frame[cat_cols].astype(str)),
            columns=cat_cols,
            index=frame.index,
        ))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]

X_train_tree = fit_transform_tree(train_data)
tree_models = {
    "LightGBM": LGBMRegressor(random_state=SEED, n_estimators=100, verbose=-1),
    "RandomForest": RandomForestRegressor(random_state=SEED, n_estimators=100),
}
for model_name, model in tree_models.items():
    model.fit(X_train_tree, y_train)
    pred_h = model.predict(transform_tree(holdout_data))
    baseline_rows.append({"model": model_name, "split": "holdout", **mp.regression_metrics(y_holdout, pred_h)})
    if test_data is not None:
        pred_t = model.predict(transform_tree(test_data))
        baseline_rows.append({"model": model_name, "split": "test", **mp.regression_metrics(y_test, pred_t)})

# Mitra current-run rows for a single comparison table.
mitra_holdout_pred = mp.predict_regression(active_predictor, holdout_data, FEATURE_COLUMNS)
baseline_rows.append({"model": f"Mitra-{active_mode}", "split": "holdout", **mp.regression_metrics(y_holdout, mitra_holdout_pred)})
if test_data is not None:
    mitra_test_pred = mp.predict_regression(active_predictor, test_data, FEATURE_COLUMNS)
    baseline_rows.append({"model": f"Mitra-{active_mode}", "split": "test", **mp.regression_metrics(y_test, mitra_test_pred)})

metrics_table = pd.DataFrame(baseline_rows)
display(metrics_table)
print("All values above are current-run tutorial metrics. Lower MAE/RMSE is better; higher R² is better.")

METRICS_JSON = WORKDIR / "tutorial_metrics.json"
METRICS_JSON.write_text(
    json.dumps(
        {
            "profile": "E2E",
            "model": mp.MODEL_ID,
            "revision": mp.PINNED_REVISION,
            "data_sha256": DATA_DIGEST,
            "selection_basis": selection_basis,
            "active_mode": active_mode,
            "pretrained_holdout": baseline_holdout,
            "pretrained_test": baseline_test,
            "finetuned_holdout": finetuned_holdout,
            "finetuned_test": finetuned_test,
            "baseline_rows": baseline_rows,
        },
        indent=2,
    )
)
print("Machine-readable metrics:", METRICS_JSON)
FIT_RUN_COMPLETED = True


## 5. Predict genuinely new rows

This optional path is separate from holdout/test evaluation. It is off by default so the sample path does not unexpectedly open an upload dialog.

Before upload, the required input contract is: all columns listed in `FEATURE_COLUMNS`; order may differ; extra identifier/context columns are allowed and preserved in `predictions.csv`; the target is not required; a pre-existing `prediction` column is rejected.

The repository's inference validator checks the schema, and the fitted AutoGluon predictor restores its training-fitted preprocessing state. It does not fit encoders/scalers from the new rows.

**Uncertainty semantics:** `prediction` is a point estimate in the target's units. This pipeline does not provide a calibrated per-row prediction interval. Use held-out error, interval-capable downstream methods, or a separate calibration procedure when uncertainty matters.


In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}

if RUN_NEW_DATA_INFERENCE:
    supplied = os.environ.get("DIMER_INFERENCE_CSV", "").strip()
    if supplied:
        csv_name = Path(supplied).name
        payload = Path(supplied).read_bytes()
    else:
        from google.colab import files
        uploaded = files.upload()
        csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith(".csv")]
        if len(csvs) != 1:
            raise RuntimeError("Upload exactly one inference CSV.")
        csv_name, payload = csvs[0]

    new_data = mp.read_csv_bytes(payload, csv_name)
    X, extra_columns = mp.validate_inference_frame(new_data, FEATURE_COLUMNS)
    if extra_columns:
        print("Extra columns preserved in output but not passed to Mitra:", extra_columns)
    predictions = mp.predict_regression(active_predictor, X, FEATURE_COLUMNS)
    prediction_output = new_data.copy()
    prediction_output["prediction"] = predictions
    PREDICTIONS_CSV = WORKDIR / "predictions.csv"
    prediction_output.to_csv(PREDICTIONS_CSV, index=False)
    display(prediction_output.head())
    print("Wrote:", PREDICTIONS_CSV)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(str(PREDICTIONS_CSV))
else:
    print("New-data inference skipped. Set RUN_NEW_DATA_INFERENCE=True to exercise it.")


## 6. Export the deployable predictor and provenance

The deployable artifact is the selected AutoGluon predictor directory, not a replacement `model.safetensors`. For Mitra, that directory contains the base model state, AutoGluon preprocessing, and the support/training context needed at inference.

**Data-governance consequence:** a predictor can therefore contain or encode information derived from training/support data. Handle, retain, disclose, and distribute the exported ZIP under the same confidentiality, licensing, and retention constraints that apply to the source data.

`tutorial_run_metadata.json` records model/revision, runtime, data identity, selection basis, adaptation configuration, and metrics. `artifact_manifest.json` enumerates every other artifact file with size and SHA-256. The ZIP digest printed at the end provides whole-archive integrity when the sender communicates it through a trusted channel.


In [ ]:
from datetime import datetime, timezone

if not globals().get("FIT_RUN_COMPLETED", False):
    raise RuntimeError("Run Step 4 successfully before exporting.")

active_path = Path(active_predictor.path)
if not active_path.exists():
    raise RuntimeError("Selected predictor path does not exist.")

run_metadata = {
    "artifact_format": mp.ARTIFACT_FORMAT,
    "artifact_format_version": mp.ARTIFACT_FORMAT_VERSION,
    "base_model": mp.MODEL_ID,
    "base_model_revision": mp.PINNED_REVISION,
    "weights_sha256": mp.WEIGHTS_SHA256,
    "config_sha256": mp.CONFIG_SHA256,
    "repository_commit": REPO_COMMIT,
    "model_source": MODEL_SOURCE,
    "autogluon_version": AUTOGLUON_VERSION,
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "python_version": sys.version.split()[0],
    "cuda_available": torch.cuda.is_available(),
    "problem_type": "regression",
    "target_column": TARGET_COLUMN,
    "features": FEATURE_COLUMNS,
    "mode": active_mode,
    "selection_basis": selection_basis,
    "seed": SEED,
    "data_source": DATA_SOURCE,
    "data_sha256": DATA_DIGEST,
    "sample_revision": SAMPLE_REVISION if using_sample else None,
    "train_rows_before_cap": cap_report["before"],
    "train_rows_used": len(train_data),
    "train_row_cap_applied": cap_report["applied"],
    "holdout_rows": len(holdout_data),
    "independent_test_rows": len(test_data) if test_data is not None else None,
    "eval_metric": EVAL_METRIC,
    "fine_tuning_requested": RUN_FINE_TUNING,
    "fine_tune_steps_requested": FINE_TUNE_STEPS if RUN_FINE_TUNING else None,
    "fine_tune_time_limit_seconds": FINE_TUNE_TIME_LIMIT if RUN_FINE_TUNING else None,
    "max_memory_usage_ratio": MAX_MEMORY_USAGE_RATIO,
    "pretrained_holdout_metrics": baseline_holdout,
    "pretrained_test_metrics": baseline_test,
    "finetuned_holdout_metrics": finetuned_holdout,
    "finetuned_test_metrics": finetuned_test,
    "prediction_uncertainty": "point predictions only; no calibrated per-prediction interval",
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
}
(active_path / "tutorial_run_metadata.json").write_text(json.dumps(run_metadata, indent=2))
manifest_path = mp.write_artifact_manifest(active_path)
print("Artifact manifest:", manifest_path)

archive_base = WORKDIR / "mitra-predictor"
Path(str(archive_base) + ".zip").unlink(missing_ok=True)
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=active_path))
archive_digest = mp.sha256_file(archive)
print("Predictor ZIP:", archive)
print("Predictor ZIP SHA-256:", archive_digest)


## 7. Fresh-boundary artifact verification

An in-memory predictor is not evidence that serialization worked. This stage uses the exact exported ZIP, extracts it into a fresh directory with archive safety limits, verifies the artifact format/version, required provenance, complete file inventory, sizes and SHA-256 digests **before deserializing**, and then reloads through `TabularPredictor.load()`.

Finally, predictions from the reloaded artifact are compared with the pre-export predictor using explicit floating-point tolerances. Passing the reload alone proves less than reproducing equivalent outputs; this cell checks both.


In [ ]:
from autogluon.tabular import TabularPredictor

RELOAD_DIR = WORKDIR / "artifact-reload"
mp.safe_extract_archive(archive, RELOAD_DIR)
verified_manifest, verified_metadata = mp.validate_artifact_directory(RELOAD_DIR)

if verified_metadata["autogluon_version"] != AUTOGLUON_VERSION:
    raise RuntimeError(
        f"Artifact requires AutoGluon {verified_metadata['autogluon_version']}; "
        f"runtime has {AUTOGLUON_VERSION}."
    )

reloaded_predictor = TabularPredictor.load(str(RELOAD_DIR))
smoke_X = holdout_data[FEATURE_COLUMNS].head(8).copy()
expected = mp.predict_regression(active_predictor, smoke_X, FEATURE_COLUMNS)
actual = mp.predict_regression(reloaded_predictor, smoke_X, FEATURE_COLUMNS)

if not np.allclose(expected, actual, rtol=1e-6, atol=1e-8):
    raise RuntimeError("Fresh-boundary verification failed: predictions changed after export/reload.")

print("✓ Artifact manifest/provenance verified before deserialization.")
print("✓ Exported predictor reloaded from fresh files.")
print("✓ Reloaded regression outputs match pre-export outputs within rtol=1e-6, atol=1e-8.")


## Interpretation, limits, and next steps

If every default-path cell ran successfully, this execution establishes that:

1. the Mitra checkpoint and config matched the repository's immutable model identity and SHA-256 values;
2. the data passed the demonstrated regression/schema checks and the split method was explicit;
3. the reported metrics and executable baselines were computed on this run's partitions;
4. the selected predictor was packaged with machine-readable provenance and a complete digest manifest; and
5. a fresh reload reproduced the original predictor's outputs within the stated numeric tolerance.

It **does not** establish that the model is accurate, calibrated, fair, robust, or safe for a production decision. Establishing those claims requires representative external/held-out data, domain-specific costs and thresholds, subgroup/drift analysis where relevant, and deployment monitoring.

Useful next experiments are to enable fine-tuning and compare independent-test behavior, switch MAE versus RMSE as the selection objective, compare random and leakage-aware pre-splits on the same data, and feed the exported ZIP into the companion `ARTIFACT-INFERENCE` notebook.

### Troubleshooting

| Failure | Meaning | Corrective action |
|---|---|---|
| dependency install changes an already-imported PyTorch | runtime module state is stale | restart and run from the top |
| model checksum mismatch | bytes are not the pinned release | re-download; do not edit expected digests |
| DIMER package manifest failure | ZIP is legacy, incomplete, or inconsistent | obtain a v1 manifested DIMER model package |
| target/schema validation failure | input violates the regression contract | correct the named columns/values before fitting |
| Mitra is skipped for memory | AutoGluon cannot safely fit/register the model | reduce rows/features or use a higher-memory runtime |
| artifact manifest/digest failure | exported/received files are incomplete or changed | reject the artifact and reproduce/retransfer it |


## AI use and provenance

This tutorial has been developed with AI assistance under human direction and review. AI attribution is authorship provenance, not independent validation. Executed checks, repository review, and reproducible outputs remain the evidence for a particular release.
